# download

In [ ]:
import requests
import re
import os


url = 'https://dicom.offis.de/download/images/ddsm/'
output_folder = "/home/hassan/repo/2d-mg-workbench/data"


r = requests.get(url)
var = r.text
expected_start = '<a href="'
expected_end = '">'
result = re.findall(rf"{expected_start}(?![\?/])(.*?){expected_end}", var)

In [ ]:
from pathlib import Path
for item in result:
    file_path = os.path.join(output_folder, item)
    if Path(file_path).is_file():
        print('exists, *skips*')
        continue
    else:
        print('doesnt, downloading')
        response = requests.get(url+item, stream=True)
        response.raise_for_status()
        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print('done')

# unzip


In [ ]:
import zipfile
import pandas as pd
from pydicom import dcmread
import numpy as np
import cv2
import re
import os

source_dir = r'/home/hassan/repo/2d-mg-workbench/data/'
unzip_dir = r'/home/hassan/repo/2d-mg-workbench/unzipped/'
label_pattern = r"^(.*?)(?=_\d|$)"
period_pattern =  r"\.([^.]+)\."

In [ ]:
#Extracting
for item in os.listdir(source_dir):
    if item.endswith(".zip"):
        file_name = source_dir + item
        zip_ref = zipfile.ZipFile(file_name)
        zip_ref.extractall(unzip_dir)


        zip_ref.close()
        


In [ ]:
central = pd.read_csv('central1024.csv')

In [ ]:
# import torch
# from torch.utils.data import Dataset
# from PIL import Image


# class MammographyDataset(Dataset):
#     def __init__(self, df, transform=None):
#         self.df = df
#         self.transform = transform
        
#         # Map string labels to numbers (0, 1, 2, 3)
#         self.label_mapping = {
#             'normals': 0,
#             'benigns': 1,
#             'benign_without_callbacks': 1, # Often merged with benigns
#             'cancers': 2
#         }

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
        
#         # 1. Read the preprocessed PNG
#         image_path = row['file_path']
#         image = Image.open(image_path).convert('RGB') # Load as PIL Image
        
#         # 2. Get target label
#         label_str = row['label']
#         label_tensor = torch.tensor(self.label_mapping[label_str], dtype=torch.long)
        
#         # 3. Apply augmentation transforms (on the fly)
#         if self.transform:
#             image = self.transform(image)
            
#         return image, label_tensor

In [ ]:
# import pandas as pd
# from sklearn.model_selection import GroupShuffleSplit
# #grouped split since spreading without grouped shuffle could promote validation overfitting to the training
# gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
# train_idx, temp_idx = next(gss1.split(central, groups=central['patient_id']))

# train_df = central.iloc[train_idx]
# val_df = central.iloc[temp_idx]

In [ ]:
# import pandas as pd
# central = pd.read_csv('central.csv')

# import torch
# from torch.utils.data import DataLoader
# import torchvision.transforms.v2 as T

# # Temporary loader without normalisation, just ToImage + ToDtype
# temp_transform = T.Compose([
#     T.ToImage(),
#     T.ToDtype(torch.float32, scale=True)
# ])
# temp_ds = MammographyDataset(df=train_df, transform=temp_transform)
# temp_loader = DataLoader(temp_ds, batch_size=32, shuffle=False, num_workers=1)

# mean = 0.0
# var = 0.0
# nb_samples = 0
# for images, _ in temp_loader:
#     # images shape: (B, 3, H, W) – all channels identical
#     # compute per-pixel mean over batch, then accumulate
#     batch_samples = images.size(0)
#     images = images.view(batch_samples, 3, -1)  # flatten H,W
#     mean += images.mean(dim=[0,2]).sum(dim=0)   # mean over batch & pixels per channel
#     var += images.var(dim=[0,2]).sum(dim=0)
#     nb_samples += batch_samples

# mean /= nb_samples
# std = torch.sqrt(var / nb_samples)
# print("DDSM mean:", mean, "std:", std)

# mean_val = mean.item()
# std_val = std.item()

# # 2. Create lists of 3 for the 3 (RGB) channels
# ddsm_mean = [mean_val, mean_val, mean_val]
# ddsm_std = [std_val, std_val, std_val]

# import torch
# #**NOTE: Vertical flips, not gonna. Incorrect anatomy might mess up pretrained models learned weights? unconmfirmed, but asummed to be the case.** 
# train_transforms = T.Compose([
#     T.ToImage(),
#     T.ToDtype(torch.float32, scale=True), #scaling to float (0.0-1.0)
#     #horizontal flip
#     T.RandomHorizontalFlip(p=0.5),

#     #slight rotation and zoom in/out
#     T.RandomAffine(degrees = 10, scale=(0.9, 1.1)),

#     # T.ColorJitter(brightness = 0.2, contrast = 0.2),

#     T.Normalize(mean=ddsm_mean, std=ddsm_std)

# ])
# val_transform = T.Compose([
#     T.ToImage(),
#     T.ToDtype(torch.float32, scale=True),
#     T.Normalize(mean=ddsm_mean, std=ddsm_std)
# ])


In [ ]:
# from torch.utils.data import DataLoader

# train_dataset = MammographyDataset(df=train_df, transform=train_transforms)
# val_dataset = MammographyDataset(df=val_df, transform=val_transform)

# # # 16/1w
# # train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=1)
# # val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=1)

# physical_batch_size = 8
# accumulation_steps = 4  # Effective batch size = 8 * 4 = 32

# train_loader = DataLoader(
#     train_dataset, 
#     batch_size=physical_batch_size, 
#     shuffle=True, 
#     num_workers=4,        # Increased from 1 for better data loading speed
#     pin_memory=True       # Crucial for fast GPU transfers
# )

# val_loader = DataLoader(
#     val_dataset, 
#     batch_size=physical_batch_size, 
#     shuffle=False, 
#     num_workers=4, 
#     pin_memory=True
# )

In [ ]:


# import numpy as np
# from pydicom import dcmread
# import matplotlib.pyplot as plt
# import cv2
# arr = train_dataset[5][0][0]

# plt.imshow(arr, cmap="gray")
# plt.show()


In [ ]:
# import torch.nn as nn
# from torchvision.models import resnet50, ResNet50_Weights

# # dl resnet50 with pretrianed weights
# model = resnet50(weights=ResNet50_Weights.DEFAULT)

In [ ]:
# # Replacing the final layer with a one that outputs 3 classes
# num_ftrs = model.fc.in_features
# model.fc = nn.Linear(num_ftrs, 3)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)    

In [ ]:
# import torch.optim as optim

# # MCC defaults
# criterion = nn.CrossEntropyLoss()

# #alts adamW, adamw_8bit
# #optimizer = optim.Adam(model.parameters(), lr=0.0001) #raw
# #optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
# def set_trainable(model, list_of_prefixes_to_train):
#     """Freeze all parameters, then unfreeze only those whose name starts with a given prefix."""
#     for param in model.parameters():
#         param.requires_grad = False

#     for name, param in model.named_parameters():
#         if any(name.startswith(prefix) for prefix in list_of_prefixes_to_train):
#             param.requires_grad = True

# set_trainable(model, ['fc'])  # only the final linear layer
# optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
#                        lr=1e-4, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='min', factor=0.5, patience=2
# )


In [ ]:
# import torch

# num_epochs = 10 

# best_val_loss = float('inf') 

# for epoch in range(num_epochs):
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     print("-" * 10)
    
#     #train
#     model.train()
#     running_loss = 0.0
#     correct_preds = 0
#     total_preds = 0
    
#     for images, labels in train_loader:
        
#         images = images.to(device)
#         labels = labels.to(device)
#         optimizer.zero_grad() # zeroing the parameter gradient to clear last batch gradients so they don't accumulate before 

        
#         # Forward pass
#         outputs = model(images)
#         loss = criterion(outputs, labels)
        
#         #calc graidents and update weights
#         loss.backward()
#         optimizer.step()
        
#         # Track statistics
#         running_loss += loss.item() * images.size(0)
#         _, predictions = torch.max(outputs, 1)
#         correct_preds += (predictions == labels).sum().item()
#         total_preds += labels.size(0)
        
#     epoch_train_loss = running_loss / len(train_loader.dataset)
#     epoch_train_acc = correct_preds / total_preds
    
#     #val
#     model.eval()  # Sets the model to evaluation mode
#     val_running_loss = 0.0
#     val_correct_preds = 0
#     val_total_preds = 0
    
#     # dont need gradient for val, so no grad to save mem
#     with torch.no_grad():
#         for images, labels in val_loader:
#             images = images.to(device)
#             labels = labels.to(device)
            
#             outputs = model(images)
#             loss = criterion(outputs, labels)
            
#             val_running_loss += loss.item() * images.size(0)
#             _, predictions = torch.max(outputs, 1)
#             val_correct_preds += (predictions == labels).sum().item()
#             val_total_preds += labels.size(0)
            
#     epoch_val_loss = val_running_loss / len(val_loader.dataset)

#     epoch_val_acc = val_correct_preds / val_total_preds
    
#     print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
#     print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")
    
#     #save best model
#     if epoch_val_loss < best_val_loss:
#         print(f"Validation loss improved ({best_val_loss:.4f} -> {epoch_val_loss:.4f}). Saving model...")
#         best_val_loss = epoch_val_loss
#         torch.save(model.state_dict(), 'best_resnet50_mammography.pth')

# print("Training Complete!")

In [ ]:
# import torch
# import torch.optim as optim
# from torch.optim.lr_scheduler import ReduceLROnPlateau

# criterion = nn.CrossEntropyLoss()

# # ---------------------------
# # Phase 1: Head‑only training
# # ---------------------------
# print("Phase 1: Training classifier head only")
# set_trainable(model, ['fc'])
# # optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
# #                        lr=1e-4, weight_decay=1e-4)
# optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
#                         lr=1e-4, weight_decay=1e-4)
# scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# num_epochs_phase1 = 10
# best_val_loss = float('inf')

# scaler = torch.amp.GradScaler('cuda')

# for epoch in range(num_epochs_phase1):
#     # --- TRAINING LOOP ---
#     model.train()
#     running_loss, correct, total = 0.0, 0, 0
    
#     optimizer.zero_grad() # Zero gradients before the first step

#     for i, (images, labels) in enumerate(train_loader):
#         images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
#         # 1. Cast forward pass to mixed precision
#         with torch.amp.autocast('cuda'):
#             outputs = model(images)
#             loss = criterion(outputs, labels)
            
#             # 2. Divide the loss by the accumulation steps
#             loss = loss / accumulation_steps
            
#         # 3. Scale the loss and backward pass
#         scaler.scale(loss).backward()
        
#         # 4. Step optimizer only after reaching the accumulation target (or end of epoch)
#         if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
#             scaler.step(optimizer)
#             scaler.update()
#             optimizer.zero_grad()

#         # Update metrics (Multiply loss back by accumulation_steps for accurate tracking)
#         running_loss += (loss.item() * accumulation_steps) * images.size(0)
#         _, preds = torch.max(outputs, 1)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

#     train_loss = running_loss / len(train_loader.dataset)
#     train_acc = correct / total

#     # --- validation loop ---
#     model.eval()
#     val_running_loss, val_correct, val_total = 0.0, 0, 0
    
#     with torch.no_grad():
#         for images, labels in val_loader:
#             images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
#             # Use AMP in validation to speed up inference and save memory
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 loss = criterion(outputs, labels)
                
#             val_running_loss += loss.item() * images.size(0)
#             _, preds = torch.max(outputs, 1)
#             val_correct += (preds == labels).sum().item()
#             val_total += labels.size(0)

#     val_loss = val_running_loss / len(val_loader.dataset)
#     val_acc = val_correct / val_total
    
#     scheduler.step(val_loss)

#     print(f"Epoch {epoch+1}/{num_epochs_phase1} | "
#           f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
#           f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save(model.state_dict(), 'best_resnet50_phase1.pth')
#         print("  -> Model saved (phase 1 best)")

# # ---------------------------
# # Load the best Phase 1 model before Phase 2
# # ---------------------------
# model.load_state_dict(torch.load('best_resnet50_phase1.pth'))
# print("\nPhase 2: Unfreezing layer4 + fc")

# # Unfreeze layer4 and fc (the rest stays frozen)
# set_trainable(model, ['fc', 'layer4'])

# # New optimizer with 10× smaller learning rate (Switched to AdamW to match Phase 1)
# optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
#                        lr=1e-5, weight_decay=1e-4)
# scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# num_epochs_phase2 = 15   # more epochs for fine‑tuning
# best_val_loss = float('inf')   # reset best loss for this phase

# for epoch in range(num_epochs_phase2):
#     # --- TRAINING LOOP ---
#     model.train()
#     running_loss, correct, total = 0.0, 0, 0
    
#     optimizer.zero_grad() # Zero gradients before the first step

#     for i, (images, labels) in enumerate(train_loader):
#         images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
#         # 1. Cast forward pass to mixed precision
#         with torch.amp.autocast('cuda'):
#             outputs = model(images)
#             loss = criterion(outputs, labels)
            
#             # 2. Divide the loss by the accumlation steps
#             loss = loss / accumulation_steps
            
#         # 3. Scale the loss and backward pass
#         scaler.scale(loss).backward()
        
#         # 4. Step optimizer only after reaching the accumulation target (or end of epoch)
#         if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
#             scaler.step(optimizer)
#             scaler.update()
#             optimizer.zero_grad()

#         # Update metrics (Multiply loss back by accumulation_steps for accurate tracking)
#         running_loss += (loss.item() * accumulation_steps) * images.size(0)
#         _, preds = torch.max(outputs, 1)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

#     train_loss = running_loss / len(train_loader.dataset)
#     train_acc = correct / total

#     # --- VALIDATION LOOP ---
#     model.eval()
#     val_running_loss, val_correct, val_total = 0.0, 0, 0
    
#     with torch.no_grad():
#         for images, labels in val_loader:
#             images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
#             # Use AMP in validation to speed up inference and save memory
#             with torch.amp.autocast('cuda'):
#                 outputs = model(images)
#                 loss = criterion(outputs, labels)
                
#             val_running_loss += loss.item() * images.size(0)
#             _, preds = torch.max(outputs, 1)
#             val_correct += (preds == labels).sum().item()
#             val_total += labels.size(0)

#     val_loss = val_running_loss / len(val_loader.dataset)
#     val_acc = val_correct / val_total
    
#     scheduler.step(val_loss)

#     print(f"Epoch {epoch+1}/{num_epochs_phase2} | "
#           f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
#           f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save(model.state_dict(), 'best_resnet50_phase2.pth')
#         print("  -> Model saved (phase 2 best)")

# print("Training Complete!")

In [ ]:
import pydicom
import numpy as np
from pydicom import dcmread
import matplotlib.pyplot as plt
import cv2

path = r"/home/hassan/repo/2d-mg-workbench/unzipped/normals/patient0000/0000.LEFT_CC.dcm"
ds = dcmread(path)
arr = pydicom.pixel_data_handlers.apply_voi_lut(ds.pixel_array, ds)
plt.imshow(arr, cmap="gray")
plt.show()

In [ ]:
second_= pydicom.pixels.pixel_array(ds)
second_

In [ ]:
plt.imshow(arr, cmap="gray")
plt.show()

In [ ]:
import pydicom
from pathlib import Path
from collections import defaultdict

root = Path("/home/hassan/repo/2d-mg-workbench/unzipped")
by_patient = defaultdict(set)

for dcm_path in root.rglob("*.dcm"):
    ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
    pid = str(ds.get("PatientID", "UNKNOWN"))
    by_patient[pid].add((
        str(ds.SOPInstanceUID),
        str(ds.get("StudyInstanceUID", "")),
        str(ds.get("StudyDate", "")),
        str(ds.get("TransferSyntaxUID", "")),
        str(ds.get("PhotometricInterpretation", "")),
        str(ds.get("WindowCenter", "")),
        str(ds.get("WindowWidth", "")),
    ))

total_images = sum(len(v) for v in by_patient.values())

# by_patient = defaultdict(set)

# for dcm_path in root.rglob("*.dcm"):
#     ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
#     pid = str(ds.get("PatientID", "UNKNOWN"))
#     study_uid = str(ds.get("StudyInstanceUID", ""))
#     study_date = str(ds.get("StudyDate", ""))
#     syn_id = str(ds.file_meta.TransferSyntaxUID.name)
#     photometric_interpretation = str(ds.PhotometricInterpretation)
#     #Window Center / Window Width
#     win_center = str(ds.WindowCenter)
#     win_width = str(ds.WindowWidth)

#     by_patient[pid].add((study_uid, study_date, syn_id, photometric_interpretation, win_center, win_width))    

# longitudinal = {pid: studies for pid, studies, in by_patient.items() if len(studies) > 1}
# print(f"{len(by_patient)} patients total, {len(longitudinal)} with >1 study")

In [ ]:
# patient_dates = {
#     pid: {item[1] for item in studies}  # item[1] is study_date
#     for pid, studies in by_patient.items()
# }

# longitudinal_by_date = {
#     pid: dates for pid, dates in patient_dates.items() if len(dates) > 1
# }

# print(f"{len(by_patient)} patients total, {len(longitudinal_by_date)} with >1 study date (Longitudinal)")

unique_syntax_ids = {item[3] for tuples_set in by_patient.values() for item in tuples_set}
print(unique_syntax_ids)

unique_photometric_interpretation = {item[4] for tuples_set in by_patient.values() for item in tuples_set}
print(unique_photometric_interpretation)

unique_win_center = [item[5] for tuples_set in by_patient.values() for item in tuples_set]
print(len(unique_win_center))

unique_win_width = [item[6] for tuples_set in by_patient.values() for item in tuples_set]
print(len(unique_win_width))
sum(len(v) for v in by_patient.values())

In [ ]:
    from datetime import datetime

def days_between(date1_str, date2_str):
    d1 = datetime.strptime(str(date1_str), "%Y%m%d")
    d2 = datetime.strptime(str(date2_str), "%Y%m%d")

    return abs((d2-d1).days)
patient_day_diffs = {}

multi_studies = {pid: studies for pid, studies in by_patient.items() if len(studies) > 1}

for pid, studies in multi_studies.items():
    studies_list = list(studies)

    date_1 = studies_list[0][1]
    date_2 = studies_list[1][1]

    diff_days = days_between(date_1, date_2)
    patient_day_diffs[pid] = diff_days
print(patient_day_diffs)


In [ ]:
res = 0 
for val in patient_day_diffs.values():
    res += val

res = res / len(patient_day_diffs)
res

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

day_diffs = list(patient_day_diffs.values())

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 5))

# Plot raw days
n, bins, patches = ax.hist(day_diffs, bins=10, linewidth=0.8, edgecolor="white", color="#2b5c8f")

# Explicit labels for Days
ax.set_xlabel("Days Between Patient Studies", fontsize=11, fontweight='bold', labelpad=10)
ax.set_ylabel("Number of Patients (Count)", fontsize=11, fontweight='bold', labelpad=10)
ax.set_title("Distribution of Time Intervals Between Follow-up Studies", fontsize=13, fontweight='bold', pad=15)

# Dynamic scaling
ax.set_xlim(left=0, right=max(day_diffs) * 1.1)
ax.set_ylim(bottom=0, top=max(n) * 1.1)

plt.tight_layout()
plt.show()


In [ ]:
import glob
from pydicom import dcmread

search_pattern = "/home/hassan/repo/2d-mg-workbench/unzipped/normals/patient00*/00*.dcm"
all_files = glob.glob(search_pattern)
all_files.sort()

files_to_process = all_files[:3300]


def diagnose(path):
    ds = dcmread(path)
    arr = ds.pixel_array.astype(np.float64)
    print(path)
    print(f"  min/max: {arr.min()}/{arr.max()}")
    print(f"  percentiles 1/50/99/99.9: {np.percentile(arr, [1,50,99,99.9])}")
    print(f"  fraction above 99th-percentile-of-typical (e.g. >250): {(arr > 250).mean():.5f}")


for path in files_to_process:
    try:
        diagnose(path=path)
    except Exception as e:
        print(f"Error reading {path}: {e}")

In [ ]:
def diagnose_source(path):
    ds = dcmread(path)
    arr = ds.pixel_array
    print(path)
    print(f"  max: {arr.max()}")
    print(f"  Manufacturer: {getattr(ds, 'Manufacturer', 'N/A')!r}")
    print(f"  ManufacturerModelName: {getattr(ds, 'ManufacturerModelName', 'N/A')!r}")
    print(f"  ImagerPixelSpacing: {getattr(ds, 'ImagerPixelSpacing', 'N/A')}")
    print(f"  StudyDescription: {getattr(ds, 'StudyDescription', 'N/A')!r}")
    print(f"  SeriesDescription: {getattr(ds, 'SeriesDescription', 'N/A')!r}")
    print(f"  SeriesInstanceUID: {getattr(ds, 'SeriesInstanceUID', 'N/A')}")
    # check for any private (odd-group) tags — conversion tools sometimes stash source info here
    for elem in ds:
        if elem.tag.is_private:
            print(f"  PRIVATE {elem.tag}: {elem.value}")

all_files = glob.glob(search_pattern)
all_files.sort()
path =  r"/home/hassan/repo/2d-mg-workbench/unzipped/normals/patient0000/0000*.dcm"


for path in files_to_process:
    try:
        
        diagnose_source(path)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        